# 0823_lsw_002_baseline

siemens_aoi 실제 데이터로 Phase 0 baseline을 만듭니다: 검사유형(`inspection_type`)별로 데이터를 분리하고, 시간순으로 Train/Val/Test를 나눈 뒤 Dummy / Logistic Regression / XGBoost를 학습하고, Validation에서 운영 임계값을 고른 다음 Test로 평가합니다.

## 1. 설정과 라이브러리

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

EXPERIMENT_ID = "0823_lsw_002_baseline"
RANDOM_STATE = 42
DATA_DIR = Path("../data/raw")
MODEL_DIR = Path("../models")

# 총비용 계산용 임시 비용 가중치. 팀 합의 전 잠정값이며 미검(FN)이 오검(FP)보다
# 훨씬 costly하다는 프로젝트 목표(Slip Rate <= 1%)를 반영해 100:1로 설정했다.
COST_FN = 100  # 미검비용: 실제 불량을 false call로 오판해 유출시켰을 때
COST_FP = 1    # 오검비용: 실제 false call을 불량으로 오판해 불필요하게 수동검사로 보냈을 때

## 2. 데이터 로딩

원본 파일은 수정하지 않고 읽기만 합니다.

In [2]:
df = pd.read_csv(DATA_DIR / "dataset.csv", index_col=0)
with open(DATA_DIR / "mapping.json", encoding="utf-8") as f:
    inspection_type_mapping = json.load(f)

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("shape:", df.shape)

shape: (440274, 77)


## 3. 평가 지표 함수 (Slip Rate / Volume Reduction / 총비용)

이후 모든 실험이 재사용할 수 있도록 함수로 고정합니다.

- `class`(0=false call, 1=true defect) 원본 라벨을 그대로 사용합니다.
- 모델 예측 1 = "수동검사(MIS)로 보낸다", 예측 0 = "자동으로 정상 처리하고 수동검사를 생략한다"로 해석합니다.
- **Slip Rate**: 실제 불량(class=1) 중 자동으로 정상 처리(예측 0)해서 유출된 비율. 낮을수록 안전합니다.
- **Volume Reduction**: 실제 false call(class=0) 중 자동으로 정상 처리(예측 0)해서 수동검사를 생략한 비율. 높을수록 검사량이 줄어듭니다.
- **총비용**: FN(놓친 불량) x `COST_FN` + FP(불필요하게 수동검사로 보낸 false call) x `COST_FP`.

In [3]:
def slip_rate(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_positive = y_true == 1
    if actual_positive.sum() == 0:
        return 0.0
    fn = ((y_pred == 0) & actual_positive).sum()
    return fn / actual_positive.sum()


def volume_reduction(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    actual_negative = y_true == 0
    if actual_negative.sum() == 0:
        return 0.0
    tn = ((y_pred == 0) & actual_negative).sum()
    return tn / actual_negative.sum()


def total_cost(y_true, y_pred, cost_fn=COST_FN, cost_fp=COST_FP):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn * cost_fn + fp * cost_fp

## 4. 검사유형별 시간순 분할

`inspection_type`으로 먼저 나눈 뒤, 각 유형 안에서 `timestamp` 기준으로 정렬해 앞 60%를 Train, 다음 20%를 Val, 마지막 20%를 Test로 사용합니다. 날짜(달력일) 기준이 아니라 행 개수 기준 분위로 자릅니다 — EDA에서 확인했듯 일별 데이터 건수 편차가 커서(12~23,046건) 달력일 기준으로 자르면 특정 구간이 비거나 너무 작아질 위험이 있기 때문입니다.

In [4]:
def chronological_split(type_df, train_frac=0.6, val_frac=0.2, timestamp_col="timestamp"):
    ordered = type_df.sort_values(timestamp_col)
    n = len(ordered)
    n_train = int(n * train_frac)
    n_val = int(n * (train_frac + val_frac))
    return ordered.iloc[:n_train], ordered.iloc[n_train:n_val], ordered.iloc[n_val:]

## 5. 피처 컬럼 선택

`mapping.json` 기준으로 해당 `inspection_type`에 유효한 컬럼만 사용합니다(무효 컬럼은 유형마다 채움값이 달라 값 기반으로 무효를 추론할 수 없다는 걸 EDA에서 확인했습니다 — `docs/lsw/siemens_aoi_v2/notes.md` 참고). 그 중에서도 **Train 데이터 기준으로 분산이 0인(상수) 컬럼은 제외**합니다 — 데이터 누수를 피하기 위해 기준을 Train에만 둡니다.

In [5]:
def get_feature_columns(inspection_type, train_df):
    valid_columns = inspection_type_mapping[str(inspection_type)]
    nunique = train_df[valid_columns].nunique()
    return nunique[nunique > 1].index.tolist()

## 6. 운영 임계값 선택

Validation에서 Slip Rate가 1% 이하를 만족하는 임계값 중 가장 높은 값을 고릅니다 (임계값이 높을수록 Volume Reduction이 커지므로, 안전 제약을 만족하는 한 가장 공격적인 임계값을 선택). 만족하는 임계값이 없으면 0.0(전부 수동검사로 보냄, Slip Rate 0%지만 Volume Reduction도 0%)으로 안전하게 fallback합니다.

In [6]:
def select_threshold(y_val, proba_val, max_slip_rate=0.01):
    for threshold in np.linspace(1.0, 0.0, 101):
        y_pred = (proba_val >= threshold).astype(int)
        if slip_rate(y_val, y_pred) <= max_slip_rate:
            return threshold
    return 0.0

## 7. 검사유형별 baseline 학습

`inspection_type`마다 Dummy / Logistic Regression / XGBoost를 각각 학습합니다 (팀과 상의한 결과, type마다 유효 피처 집합과 무효 컬럼 채움값 패턴이 달라 단일 모델보다 유형별 모델을 baseline으로 우선 채택했습니다). 클래스 불균형에 대한 명시적 리샘플링/가중치 조정은 이번 baseline에서는 적용하지 않고(향후 Phase 1에서 다룸), 임계값 조정만으로 비용에 대응합니다.

In [7]:
model_factories = {
    "dummy": lambda: DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "logistic_regression": lambda: Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
        ]
    ),
    "xgboost": lambda: XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

results = []
fitted_models = {}

for inspection_type in sorted(df["inspection_type"].unique()):
    type_df = df[df["inspection_type"] == inspection_type]
    train_df, val_df, test_df = chronological_split(type_df)
    feature_columns = get_feature_columns(inspection_type, train_df)

    X_train, y_train = train_df[feature_columns], train_df["class"]
    X_val, y_val = val_df[feature_columns], val_df["class"]
    X_test, y_test = test_df[feature_columns], test_df["class"]

    for model_name, model_factory in model_factories.items():
        model = model_factory()
        model.fit(X_train, y_train)

        proba_val = model.predict_proba(X_val)[:, 1]
        threshold = select_threshold(y_val, proba_val)

        proba_test = model.predict_proba(X_test)[:, 1]
        y_pred_test = (proba_test >= threshold).astype(int)

        results.append(
            {
                "inspection_type": inspection_type,
                "model": model_name,
                "n_train": len(train_df),
                "n_val": len(val_df),
                "n_test": len(test_df),
                "n_features": len(feature_columns),
                "threshold": threshold,
                "test_slip_rate": slip_rate(y_test, y_pred_test),
                "test_volume_reduction": volume_reduction(y_test, y_pred_test),
                "test_total_cost": total_cost(y_test, y_pred_test),
            }
        )
        fitted_models[(inspection_type, model_name)] = (model, feature_columns, threshold)

results_df = pd.DataFrame(results)
results_df

,inspection_type,model,n_train,n_val,n_test,n_features,threshold,test_slip_rate,test_volume_reduction,test_total_cost
0,0,dummy,58231,19411,19411,37,0.0,0.0,0.0,19220
1,0,logistic_regression,58231,19411,19411,37,0.0,0.0,0.0,19220
2,0,xgboost,58231,19411,19411,37,0.0,0.0,0.0,19220
3,1,dummy,34603,11535,11535,24,0.0,0.0,0.0,10798
4,1,logistic_regression,34603,11535,11535,24,0.0,0.0,0.0,10798
5,1,xgboost,34603,11535,11535,24,0.0,0.0,0.0,10798
6,2,dummy,76904,25635,25635,20,0.0,0.0,0.0,24885
7,2,logistic_regression,76904,25635,25635,20,0.0,0.0,0.0,24885
8,2,xgboost,76904,25635,25635,20,0.0,0.0,0.0,24885
9,3,dummy,91158,30386,30387,19,0.0,0.0,0.0,29804


## 8. 결과 요약

In [8]:
summary = results_df.pivot_table(
    index="inspection_type",
    columns="model",
    values=["test_slip_rate", "test_volume_reduction", "test_total_cost"],
)
summary

test_slip_rate                             test_total_cost  \
model                    dummy logistic_regression xgboost           dummy   
inspection_type                                                              
0                          0.0                 0.0     0.0         19220.0   
1                          0.0                 0.0     0.0         10798.0   
2                          0.0                 0.0     0.0         24885.0   
3                          0.0                 0.0     0.0         29804.0   
4                          0.0                 0.0     0.0          1009.0   

                                             test_volume_reduction  \
model           logistic_regression  xgboost                 dummy   
inspection_type                                                      
0                           19220.0  19220.0                   0.0   
1                           10798.0  10798.0                   0.0   
2                           24885.0  24885.0                   0.0   
3                           29804.0  29804.0                   0.0   
4                            1009.0   1009.0                   0.0   

                                             
model           logistic_regression xgboost  
inspection_type                              
0                               0.0     0.0  
1                               0.0     0.0  
2                               0.0     0.0  
3                               0.0     0.0  
4                               0.0     0.0

## 9. 모델 저장

로드맵상 XGBoost가 주력 모델이므로, 검사유형별 XGBoost 모델과 선택된 임계값·피처 목록을 함께 저장합니다. 모델 바이너리는 Git에서 제외됩니다.

In [9]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for inspection_type in sorted(df["inspection_type"].unique()):
    model, feature_columns, threshold = fitted_models[(inspection_type, "xgboost")]
    model_path = MODEL_DIR / f"{EXPERIMENT_ID}_type{inspection_type}.pkl"
    joblib.dump(
        {"model": model, "feature_columns": feature_columns, "threshold": threshold},
        model_path,
    )
    print(f"saved: {model_path}")

saved: ..\models\0823_lsw_002_baseline_type0.pkl
saved: ..\models\0823_lsw_002_baseline_type1.pkl
saved: ..\models\0823_lsw_002_baseline_type2.pkl
saved: ..\models\0823_lsw_002_baseline_type3.pkl
saved: ..\models\0823_lsw_002_baseline_type4.pkl


## 10. 결론 및 다음 단계

### 핵심 결과

검사유형(5개) x 모델(Dummy/Logistic Regression/XGBoost) 15개 조합 **전부** 선택된 임계값이 0.0으로 떨어졌습니다. 즉 Validation에서 Slip Rate ≤ 1% 제약을 만족하는 임계값이 0.0(=모든 행을 수동검사로 보낸다)밖에 없었고, 그 결과 Test에서 Slip Rate 0%는 달성하지만 **Volume Reduction도 전부 0%**입니다 — 지금 baseline은 "AOI가 불량 판정한 걸 전부 그대로 수동검사로 보내는" 현재 운영과 동일해서, 아직 어떤 자동화 효과도 없습니다. 목표 지표(Slip Rate ≤ 1%, Volume Reduction ≥ 40%) 중 Slip Rate는 자명하게 달성했지만 Volume Reduction 목표는 전혀 달성하지 못했습니다.

### 원인 진단 (type3, XGBoost로 직접 확인)

이게 모델이 나빠서가 아님을 확인했습니다. type3 XGBoost의 Validation AUC는 0.915로 양호합니다. 문제는:

- type3 Validation의 실제 불량(class=1) 표본이 53개뿐이라, Slip Rate ≤ 1%를 만족하려면 `0.01 x 53 = 0.53` → **놓쳐도 되는 개수가 사실상 0개**입니다(1개만 놓쳐도 1/53 ≈ 1.9%로 초과).
- 53개 중 15개는 모델이 예측한 확률이 0.01 미만이고, 가장 낮은 건 확률 0.0000115(사실상 0)입니다. 이 한두 개의 "극단적으로 낮게 예측된" 실제 불량 때문에, 그걸 잡으려면 임계값을 거의 0까지 낮춰야 하고 그러면 거의 모든 행이 "수동검사 필요"로 분류되어 Volume Reduction이 0이 됩니다.
- 표본 수가 더 적은 type0/1/4는 이 문제가 더 심합니다(type4는 Validation 불량이 20개 미만).

### 다음 단계 제안 (팀/사용자 판단 필요)

1. **Slip Rate를 유형별이 아니라 전체 데이터 합산으로 평가하는 방식 검토.** 유형별로 쪼개면 표본이 작아 단 1건의 오분류가 비율을 크게 흔듭니다. 전체로 합치면 표본이 커져(전체 불량 4,622건) 통계적으로 더 안정적인 임계값 선택이 가능할 수 있습니다.
2. 위 15개 중 임계값이 0에 가깝게 몰리는 "극단적으로 낮게 예측된 실제 불량" 샘플들을 직접 들여다보고, 라벨 노이즈(로드맵 Phase 3)인지 실제로 애매한 케이스인지 확인.
3. Slip Rate ≤ 1%가 정말 유형별 단위 제약인지, 전체 파이프라인 단위 제약인지 팀과 다시 확인.
4. 위 결정 전까지는 이 baseline을 "Volume Reduction 0%"라는 출발점으로 놓고, Phase 1(불균형 대응)·임계값 정책 변경 실험과 비교하는 기준선으로 사용.
